## Limpieza products 
Objetivos del notebook: 
- Limpiar los datos de productos provenientes de la capa bronce.
- Estandarizar y limpiar los datos. 
- Mandar a cuarentena aquellos datos que requieren revision más profunda. 
- Cargar los datos en la capa plata.

### Resumen Columnas
|       Campo       |                        Descripción                        |
|-------------------|-----------------------------------------------------------|
| `product_id`      | Identificación del producto                               |
| `name`            | Nombre del Producto                                       |
| `brand`           | Marca del producto                                        |
| `category`        | Struct `{l1, l2}`: categoría y subcategoría               |
| `price`           | Struct `{base_usd, currency}`: precio de lista en USD     |
| `attributes`      | Struct con atributos **opcionales** que cambian según la categoría (color, peso, talla, garantía…) |
| `tags`            | Array de etiquetas                                        |
| `suppliers`       | Array de structs `{supplier_id, cost_usd, lead_time_days, country_origin, is_preferred}` |
| `active`          | Valor booleano que define si el producto es activo o inactivo |
| `created_at`      | Fecha de creacion de ese registro (fecha como texto ISO-8601) |
| `discontinued_at` | Fecha de descontinuacion          (fecha como texto ISO-8601) |
| `dlh_hash_products` | Hash de products generado por todas las columnas sin el prefijo `dlh_`     |
| `dlh_carga_id`     | Identificador de la carga a la que pertenece.             |
| `dlh_fecha_ingesta_bronce`| Fecha de ingesta en la capa bronce                 |
| `dlh_fecha_ingesta`| Fecha de ingesta en la capa plata                         |

## Librerias y datos

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "01-funciones-compartidas"))

from pyspark.sql import functions as F, types as T, Window as W
from delta.tables import DeltaTable
from funciones_limpieza import limpiar_texto, estandarizar_columnas_str, fecha_de_corte_data, generar_hash
from funciones_conexion import obtener_ruta
from funciones_actualizacion_marcas import obtener_ventana_fechas_carga_id, actualizar_fechas_id_carga

## Ingesta de Datos

In [0]:
# 1. Lectura de la tabla de Bronce
schema = "bronce"
tabla = "products"
schema_actualizacion = "plata"
bronce_products = spark.table(
    "dlh_nova_shop."+schema+"."+tabla
)

# 2. Valores de control de fechas y carga_id 
ultima_fecha_actualizacion, maxima_fecha_bronce,resumen_carga,ventana_carga_id, bronce_products = obtener_ventana_fechas_carga_id(
    spark= spark,
    tabla_df = bronce_products,
    nombre_tabla=tabla,
    nombre_schema_tabla_actualizacion= schema_actualizacion
    )
# 3. Resumen de valores 
print(f"Watermark anterior: {ultima_fecha_actualizacion}")
print(f"Fecha máxima de ingesta en bronce: {maxima_fecha_bronce}")
print(f"Registros a procesar en esta carga incremental: {bronce_products.count()}")
print(f"Cargas a reemplazar (REPLACE WHERE): {ventana_carga_id}")

## Limpieza de Datos

### Limpieza product_id

In [0]:
bronce_products = bronce_products.withColumn(
    "product_id", 
    F.upper(F.trim(F.col("product_id")))
    ).withColumn(
        "product_id",
        F.when(
            F.col("product_id").rlike(r"^P\d{4}$"),
            F.col("product_id")
        )
        .otherwise("N/A")
    )

### Limpieza name

In [0]:
bronce_products = bronce_products.withColumn(
    "name",
    limpiar_texto(F.col("name"))
)

### Limpieza brand

In [0]:
bronce_products = bronce_products.withColumn(
    "brand",
    limpiar_texto(F.col("brand"))
    )

###  Limpieza category

In [0]:
bronce_products = bronce_products.withColumn(
        "category",
        F.col("category").withField(
            "l1",
            limpiar_texto(F.col("category.l1"))
        ).withField(
            "l2",
            limpiar_texto(F.col("category.l2"))
        )
    )

### Limpieza price

In [0]:
bronce_products = bronce_products.withColumn(
    "price",
    F.col("price").withField(
        "base_usd",
        F.abs(F.col("price.base_usd")).cast("decimal(10,2)")
    ).withField(
        "currency",
        F.upper(F.trim(F.col("price.currency")))
    )
)

### Limpieza attributes

In [0]:
bronce_products = bronce_products.withColumn(
    "attributes",
    F.col("attributes")
        .withField(
            "battery_required",
            F.col("attributes.battery_required").cast("boolean")
        ).withField(
            "color",
            limpiar_texto(F.col("attributes.color"))
        # Estandariza el formato del texto y reemplaza valores nulos
        ).withField(
            "connectivity",
            F.when(
                F.col("attributes.connectivity").isNull() |
                F.upper(F.trim(F.col("attributes.connectivity"))).isin("NULL", "N/A", "-", ""),
                F.lit("N/A")
            ).otherwise(
                F.when(
                    F.initcap(F.trim(F.col("attributes.connectivity"))) == "Wifi",
                    "WIFI"
                ).when(
                    F.initcap(F.trim(F.col("attributes.connectivity"))) == "Usb",
                    "USB"
                ).when(
                    F.initcap(F.trim(F.col("attributes.connectivity"))) == "Ninguna",
                    "N/A"
                ).otherwise(
                    F.initcap(F.trim(F.col("attributes.connectivity")))
                )
            )
        ).withField(
            "material",
            limpiar_texto(F.col("attributes.material"))
        ).withField(
            "min_age_years",
            F.abs(F.col("attributes.min_age_years")).cast("integer")
        ).withField(
            "organic",
            F.col("attributes.organic").cast("boolean")
        ).withField(
            "shelf_life_days",
            F.abs(F.col("attributes.shelf_life_days")).cast("integer")
        ).withField(
            "size",
            F.when(
                F.col("attributes.size").isNull() |
                F.upper(F.trim(F.col("attributes.size"))).isin("NULL", "N/A", "-", ""),
                F.lit("N/A")
            ).otherwise(
                F.upper(F.trim(F.col("attributes.size")))
            )
        ).withField(
            "vegan",
            F.col("attributes.vegan").cast("boolean")
        ).withField(
            "volume_ml",
            F.abs(F.col("attributes.volume_ml")).cast("integer")
        ).withField(
            "warranty_months",
            F.abs(F.col("attributes.warranty_months")).cast("integer")
        ).withField(
            "weight_kg",
            F.abs(F.col("attributes.weight_kg")).cast("decimal(10,2)")
        )
)


### Limpieza tags

In [0]:
bronce_products = bronce_products.withColumn(
        "tags",
        F.coalesce(
            F.array_distinct(
                F.transform(
                    F.col("tags"),
                    lambda tag: F.initcap(F.trim(tag))
                )
            ),
            F.array().cast("array<string>")
        )
    )

### Limpieza suppliers

In [0]:
bronce_products = bronce_products.withColumn(
    "suppliers",
    F.transform(
        F.col("suppliers"),
        lambda x: 
            x.withField(
               "cost_usd",
               F.abs(x["cost_usd"])
            ).withField(
                "country_origin",
                F.upper(F.trim(x["country_origin"]))
            ).withField(
                "is_preferred",
                x["is_preferred"].cast("boolean")
            ).withField(
                "lead_time_days",
                F.abs(x["lead_time_days"]).cast("integer")
            ).withField(
                "supplier_id",
                F.when(
                    F.upper(F.trim(x["supplier_id"])).rlike(r"^S\d{3}$"),
                    F.upper(F.trim(x["supplier_id"]))
                ).otherwise(
                    "N/A"
                )
            )
    )
)

### Limpieza active

In [0]:
bronce_products = bronce_products.withColumn(
    "active",
    F.col("active").cast("boolean")
)

### Limpieza created_at

In [0]:
bronce_products = bronce_products.withColumn(
    "created_at",
    F.try_to_timestamp(F.col("created_at"),F.lit("yyyy-MM-dd'T'HH:mm:ssX"))
)

### limpieza discontinued_at

In [0]:
bronce_products = bronce_products.withColumn(
    "discontinued_at",
    F.try_to_timestamp(F.col("discontinued_at"),F.lit("yyyy-MM-dd'T'HH:mm:ssX"))
)

## Limpieza clientes por logica

In [0]:
bronce_products = bronce_products.withColumn(
    "fecha_de_corte",
    fecha_de_corte_data()
)

In [0]:
bronce_products = bronce_products.dropDuplicates()

### product_id N/A

In [0]:
C_sin_product_id = bronce_products.filter(
    F.col("product_id") == "N/A",
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("product_id N/A")
)
bronce_products = bronce_products.filter(
    F.col("product_id") != "N/A"
)

### Cuarentena por Fechas

#### created_at nulo

In [0]:
C_sin_created_at = bronce_products.filter(
    F.col("created_at").isNull()
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("created_at is NULL")
)

bronce_products = bronce_products.filter(
    F.col("created_at").isNotNull()
)

#### Estado activo cuando ya tiene fecha discontinued_at

In [0]:
C_activo_con_discontinued_at = bronce_products.filter(
    (F.col("active")==True)
    & (F.col("discontinued_at").isNotNull())
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("active True CON FECHA DE discontinued_at")
)

bronce_products = bronce_products.filter(
    (
        (F.col("active") == True)
        & (F.col("discontinued_at").isNull())
    )
    | (F.col("active") == False)
)

#### Estado inactivo sin fecha de discontinued_at

In [0]:
C_inactivo_sin_discontinued_at = bronce_products.filter(
    ((F.col("active") == False)
    & (F.col("discontinued_at").isNull()))
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("active False SIN FECHA DE discontinued_at")
)

bronce_products = bronce_products.filter(
    (   
        (F.col("active") == False)
        & (F.col("discontinued_at").isNotNull())
    )
    | (F.col("active") == True)
)


#### Fechas de created_at y discontinued_at

In [0]:
C_created_at_mayor_discontinued_at = bronce_products.filter(
    (F.col("discontinued_at").isNotNull())
    & (F.col("created_at") > F.col("discontinued_at")),
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("created_at > discontinued_at")
)

bronce_products = bronce_products.filter(
    (F.col("discontinued_at").isNull())
    | (F.col("created_at") <= F.col("discontinued_at")),
)

In [0]:
C_discontinued_at_mayor_fecha_corte = bronce_products.filter(
    (F.col("discontinued_at").isNotNull())
    & (F.col("discontinued_at") > F.col("fecha_de_corte")),
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("discontinued_at > fecha_de_corte")
)

bronce_products = bronce_products.filter(
    (F.col("discontinued_at").isNull() )
    | (F.col("discontinued_at") <= F.col("fecha_de_corte")),
)

## Exportando Tabla

In [0]:
#   .option("path","RUTA ADLSG2") \
cuarentena_products = C_discontinued_at_mayor_fecha_corte \
    .unionByName(C_sin_product_id, allowMissingColumns=True) \
    .unionByName(C_sin_created_at, allowMissingColumns=True) \
    .unionByName(C_created_at_mayor_discontinued_at, allowMissingColumns=True) \
    .unionByName(C_activo_con_discontinued_at, allowMissingColumns=True) \
    .unionByName(C_inactivo_sin_discontinued_at, allowMissingColumns=True)

# REPLACE WHERE: borra y reinserta SOLO las filas de las cargas de esta ventana.
# Si la ejecución falla a medias, la re-ejecución reemplaza las mismas filas (sin duplicados)
if ventana_carga_id is not None:
    cuarentena_products.write \
        .mode("overwrite") \
        .format("delta") \
        .option("replaceWhere", ventana_carga_id) \
        .option("delta.feature.allowColumnDefaults", "supported") \
        .saveAsTable("dlh_nova_shop.cuarentena.products")
else:
    print("Bronce vacio: no hay registros que escribir en cuarentena")

In [0]:
columnas_hash = [
    "product_id","name", "brand", "category", "price", "attributes", "tags", "suppliers","active", "created_at", "discontinued_at"
]
plata_products = generar_hash(bronce_products,columnas_hash,"dlh_hash_products")

plata_products = plata_products.select(
    F.col("product_id"),
    F.col("name"),
    F.col("brand"),
    F.col("category"),
    F.col("price"),
    F.col("attributes"),
    F.col("tags"),
    F.col("suppliers"),
    F.col("active"),
    F.col("created_at"),
    F.col("discontinued_at"),
    F.col("dlh_hash_products"),
    F.col("dlh_carga_id"),
    F.col("dlh_fecha_ingesta").alias("dlh_fecha_ingesta_bronce"),
    F.current_timestamp().alias("dlh_fecha_ingesta")
)

In [0]:
plata_products = plata_products.withColumn(
    "dedup_hash",
    F.row_number().over(
        W.partitionBy(
            F.col("dlh_hash_products")
        )
    )
).filter(
    F.col("dedup_hash") == 1
).drop("dedup_hash")

In [0]:
# REPLACE WHERE: borra y reinserta SOLO las filas de las cargas de esta ventana
if ventana_carga_id is not None:
    DeltaTable.forName(spark, "dlh_nova_shop.plata.products").alias("recur").merge(
        plata_products.alias("incre"),
        "recur.dlh_hash_products == incre.dlh_hash_products"
    ).whenNotMatchedInsertAll().whenMatchedUpdateAll().execute()
else:
    print("Bronce vacio: no hay registros que escribir en plata")

## Actualizacion de tabla de control

In [0]:
actualizar_fechas_id_carga(
    spark= spark,
    resumen_carga= resumen_carga,
    maxima_fecha= maxima_fecha_bronce,
    ultima_fecha_actualizacion = ultima_fecha_actualizacion,
    tabla = tabla,
    schema = schema_actualizacion 
)